# BVNS + RL — Territory Design Problem

## Pipeline
```
TGraph pipeline:
  Train  → planar500 G0-G9    (model_tgraph.pt)
  Eval   → planar500 G0-G9  +  planar600 G0-G9

GGraph pipeline:
  Train  → Center486 G0-G9   (model_ggraph.pt)
  Eval   → Center486 G0-G9  +  Corners486 G0-G9  +  Diagonal486 G0-G9
```

## Params (match VNS baseline)
| Param | Value |
|---|---|
| `llambda` | 0.7 |
| `delta` | 0.05 |
| `nr_districts` | 10 |
| `nrInitSolutions` | 100 |
| `startingLambda` | 0.1 |
| `shaking_steps` | 25 |
| `fail_max` | 50 |

## 0. Imports & Config

In [ ]:
import os, json, time, warnings
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from rl import TerritoryDesignProblem, BVNS

warnings.filterwarnings('ignore')

# ── paths ────────────────────────────────────────────────────────
TGRAPH_INST  = 'TGraphInstances'
GGRAPH_INST  = 'GGraphInstances/newGeneratedInstances/27x27Graphs'
VNS_PLANAR   = 'Results/VNSExperiments'
VNS_GGRAPH   = 'Results/VNSExperimentsGeneral/27x27Graphs'
OUT_PLANAR   = 'Results/MyExperiments'
OUT_GGRAPH   = 'Results/MyExperimentsGeneral/27x27Graphs'
MODEL_TGRAPH = 'model_tgraph.pt'
MODEL_GGRAPH = 'model_ggraph.pt'

os.makedirs(OUT_PLANAR, exist_ok=True)
os.makedirs(OUT_GGRAPH, exist_ok=True)

# ── params (match VNS) ───────────────────────────────────────────
LMBDA        = 0.7
DELTA        = 0.05
NR_DISTRICTS = 10
NR_INIT      = 100   # VNS dùng 100
START_LAM    = 0.1   # VNS dùng 0.1
LAM_RANGE    = [0.1, 0.4, 0.7, 1.0]
SHAKING      = 25
FAIL_MAX     = 50

print('Setup OK')

## 1. Data Exploration

In [ ]:
def graph_stats(fp):
    G = nx.read_graphml(fp)
    attrs = ['demand', 'workload', 'n_customers']
    totals = {a: sum(float(G.nodes[n].get(a, 0)) for n in G.nodes) for a in attrs}
    return G.number_of_nodes(), G.number_of_edges(), totals

print(f"{'Instance':<24} {'Nodes':>6} {'Edges':>7}  demand   workload  customers")
print('-' * 72)

for size in [500, 600]:
    for i in range(3):  # show G0-G2 sample
        fp = f'{TGRAPH_INST}/planar{size}_G{i}.graphml'
        if os.path.exists(fp):
            n, e, t = graph_stats(fp)
            print(f"planar{size}_G{i:<14} {n:>6} {e:>7}  {t['demand']:>8.0f} {t['workload']:>9.0f} {t['n_customers']:>10.0f}")
print('...')
for tp in ['Center486', 'Corners486', 'Diagonal486']:
    for i in range(2):
        fp = f'{GGRAPH_INST}/{tp}_G{i}.graphml'
        if os.path.exists(fp):
            n, e, t = graph_stats(fp)
            print(f"{tp}_G{i:<13} {n:>6} {e:>7}  {t['demand']:>8.0f} {t['workload']:>9.0f} {t['n_customers']:>10.0f}")

In [ ]:
# So sánh cấu trúc TGraph vs GGraph
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

samples = [
    (f'{TGRAPH_INST}/planar500_G0.graphml',       'TGraph: planar500_G0'),
    (f'{TGRAPH_INST}/planar600_G0.graphml',       'TGraph: planar600_G0'),
    (f'{GGRAPH_INST}/Center486_G0.graphml',       'GGraph: Center486_G0'),
    (f'{GGRAPH_INST}/Diagonal486_G0.graphml',     'GGraph: Diagonal486_G0'),
]
for ax, (fp, title) in zip(axes, samples):
    G   = nx.read_graphml(fp)
    pos = {n: (float(d['x']), float(d['y'])) for n, d in G.nodes(data=True)}
    nx.draw(G, pos, ax=ax, node_size=6, node_color='steelblue',
            edge_color='#bbb', width=0.4, with_labels=False)
    ax.set_title(f'{title}\n({G.number_of_nodes()} nodes)', fontsize=9)

plt.suptitle('Graph structures: TGraph (planar random) vs GGraph (27x27 grid)', fontsize=11)
plt.tight_layout()
plt.show()

## 2. Helper Functions

In [ ]:
def make_tdp(G):
    return TerritoryDesignProblem(
        graph_input=G, delta=DELTA, llambda=LMBDA,
        rcl_parameter=0.2, nr_districts=NR_DISTRICTS
    )

def make_bvns(tdp):
    return BVNS(
        tdp_instance=tdp,
        shaking_steps=SHAKING,
        fail_max=FAIL_MAX,
        nrInitSolutions=NR_INIT,
        startingLambda=START_LAM,
        lambdaRange=LAM_RANGE,
    )

def train_on_instance(file_path, model_path):
    """Train RL agent on one instance; save updated model. No result saved."""
    G    = nx.read_graphml(file_path)
    tdp  = make_tdp(G)
    bvns = make_bvns(tdp)
    bvns.agent.load(model_path)
    bvns.performBVNS(train_agent=True)
    bvns.agent.save(model_path)

def eval_instance(file_path, instance_name, save_dir, model_path):
    """Eval RL agent (no training); save result JSON. Returns result dict."""
    t0   = time.time()
    G    = nx.read_graphml(file_path)
    tdp  = make_tdp(G)
    bvns = make_bvns(tdp)
    bvns.agent.load(model_path)
    obj_hist, inf_hist, best_sol, _ = bvns.performBVNS(train_agent=False)
    elapsed = time.time() - t0

    result = {
        'Instance':      instance_name,
        'Objective':     obj_hist[-1],
        'Infeasibility': inf_hist[-1],
        'Time Taken':    elapsed,
        'nrDistricts':   len(best_sol),
        'delta':         DELTA,
        'llambda':       LMBDA,
        'Allocation':    {str(k): [str(n) for n in v]
                          for k, v in best_sol.items()},
        '_obj_hist':     obj_hist,   # for convergence plot (not saved to JSON)
        '_inf_hist':     inf_hist,
    }
    os.makedirs(save_dir, exist_ok=True)
    with open(os.path.join(save_dir, f'{instance_name}.json'), 'w') as f:
        json.dump({k: v for k, v in result.items() if not k.startswith('_')},
                  f, indent=4)
    return result

def load_json(path):
    return json.load(open(path)) if os.path.exists(path) else None

print('Helpers defined.')

---
# TGraph Pipeline
Train model trên **planar500**, sau đó eval trên **planar500 + planar600**.

> Lý do: planar500 và planar600 cùng là đồ thị phẳng ngẫu nhiên (cùng phân phối, khác kích thước).  
> Model học được chiến lược tuning λ và shaking step k trên 500-node graphs, sau đó generalize sang 600-node.

## 3. TGraph — Training (planar500 G0–G9)

In [ ]:
# Reset model TGraph
if os.path.exists(MODEL_TGRAPH):
    os.remove(MODEL_TGRAPH)
    print(f'Removed old {MODEL_TGRAPH}')

print('Training on planar500 G0-G9 ...')
for i in range(10):
    fp = f'{TGRAPH_INST}/planar500_G{i}.graphml'
    t0 = time.time()
    print(f'  [{i+1}/10] planar500_G{i} ...', end=' ', flush=True)
    train_on_instance(fp, MODEL_TGRAPH)
    print(f'done ({time.time()-t0:.0f}s)')

print(f'\nTGraph model saved -> {MODEL_TGRAPH}')

## 4. TGraph — Evaluation (planar500 + planar600)

In [ ]:
tgraph_results = {}
print(f"{'#':<4} {'Instance':<20} {'Obj':>8} {'Inf':>8} {'Time':>7}")
print('-' * 52)
idx = 0

for size in [500, 600]:
    for i in range(10):
        name = f'planar{size}_G{i}'
        fp   = f'{TGRAPH_INST}/{name}.graphml'
        if not os.path.exists(fp):
            continue
        idx += 1
        r = eval_instance(fp, name, OUT_PLANAR, MODEL_TGRAPH)
        tgraph_results[name] = r
        print(f"{idx:<4} {name:<20} {r['Objective']:>8.3f} {r['Infeasibility']:>8.4f} {r['Time Taken']:>6.0f}s")

print(f'\nTGraph eval done: {len(tgraph_results)} instances.')

---
# GGraph Pipeline
Train model riêng trên **Center486 G0–G9**, sau đó eval trên **Center + Corners + Diagonal**.

> Lý do: GGraph là đồ thị lưới 27×27 — cấu trúc hoàn toàn khác TGraph.  
> Model train trên Center (centroid ở giữa) rồi test trên Corners/Diagonal (centroid ở góc/đường chéo)  
> → đánh giá khả năng generalize giữa các topology khác nhau trong GGraph.

## 5. GGraph — Training (Center486 G0–G9)

In [ ]:
# Reset model GGraph
if os.path.exists(MODEL_GGRAPH):
    os.remove(MODEL_GGRAPH)
    print(f'Removed old {MODEL_GGRAPH}')

print('Training on Center486 G0-G9 ...')
for i in range(10):
    fp = f'{GGRAPH_INST}/Center486_G{i}.graphml'
    t0 = time.time()
    print(f'  [{i+1}/10] Center486_G{i} ...', end=' ', flush=True)
    train_on_instance(fp, MODEL_GGRAPH)
    print(f'done ({time.time()-t0:.0f}s)')

print(f'\nGGraph model saved -> {MODEL_GGRAPH}')

## 6. GGraph — Evaluation (Center + Corners + Diagonal 486)

In [ ]:
ggraph_results = {}
print(f"{'#':<4} {'Instance':<24} {'Obj':>8} {'Inf':>8} {'Time':>7}")
print('-' * 56)
idx = 0

for tp in ['Center486', 'Corners486', 'Diagonal486']:
    for i in range(10):
        name = f'{tp}_G{i}'
        fp   = f'{GGRAPH_INST}/{name}.graphml'
        if not os.path.exists(fp):
            continue
        idx += 1
        r = eval_instance(fp, name, OUT_GGRAPH, MODEL_GGRAPH)
        ggraph_results[name] = r
        print(f"{idx:<4} {name:<24} {r['Objective']:>8.3f} {r['Infeasibility']:>8.4f} {r['Time Taken']:>6.0f}s")

print(f'\nGGraph eval done: {len(ggraph_results)} instances.')

---
# So sánh BVNS+RL vs VNS

## 7. Bảng chi tiết

In [ ]:
import pandas as pd
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_rows', 60)

def compare(rl_dir, vns_dir, names, vns_suffix='_vns'):
    rows = []
    for name in names:
        rl  = load_json(f'{rl_dir}/{name}.json')
        vns = load_json(f'{vns_dir}/{name}{vns_suffix}.json')
        if not (rl and vns):
            continue
        rows.append({
            'Instance':  name,
            'RL_Obj':    round(rl['Objective'],     3),
            'VNS_Obj':   round(vns['Objective'],    3),
            'Diff_Obj':  round(rl['Objective'] - vns['Objective'], 3),
            'RL_Inf':    round(rl['Infeasibility'],  4),
            'VNS_Inf':   round(vns['Infeasibility'], 4),
            'RL_Time':   round(rl['Time Taken'],    1),
            'VNS_Time':  round(vns['Time Taken'],   1),
        })
    return pd.DataFrame(rows)

tg_names = [f'planar{sz}_G{i}' for sz in [500,600] for i in range(10)]
gg_names = [f'{tp}_G{i}' for tp in ['Center486','Corners486','Diagonal486'] for i in range(10)]

df_t = compare(OUT_PLANAR, VNS_PLANAR, tg_names)
df_g = compare(OUT_GGRAPH, VNS_GGRAPH, gg_names)
df_all = pd.concat([df_t, df_g], ignore_index=True)

print('=== TGraph ===')
display(df_t)
print('\n=== GGraph ===')
display(df_g)

## 8. Tổng hợp theo nhóm

In [ ]:
def print_summary(df, label):
    if df.empty:
        return
    n         = len(df)
    rl_wins   = (df['Diff_Obj'] < 0).sum()
    vns_wins  = (df['Diff_Obj'] > 0).sum()
    ties      = (df['Diff_Obj'] == 0).sum()
    print(f"\n{'='*58}")
    print(f"  {label}  (n={n})")
    print(f"{'='*58}")
    print(f"  {'Metric':<16} {'BVNS+RL':>10} {'VNS':>10} {'Diff':>10}")
    print(f"  {'-'*48}")
    for col, rc, vc in [('Objective','RL_Obj','VNS_Obj'),
                         ('Infeasibility','RL_Inf','VNS_Inf'),
                         ('Time (s)','RL_Time','VNS_Time')]:
        rm, vm = df[rc].mean(), df[vc].mean()
        print(f"  {col:<16} {rm:>10.3f} {vm:>10.3f} {rm-vm:>+10.3f}")
    pct_better = rl_wins / n * 100
    print(f"\n  RL wins: {rl_wins}/{n} ({pct_better:.0f}%)  "
          f"VNS wins: {vns_wins}/{n}  Tie: {ties}/{n}")
    print(f"  Negative Diff → RL better, Positive → VNS better")

print_summary(df_t[df_t['Instance'].str.startswith('planar500')], 'TGraph — planar500 (in-distribution)')
print_summary(df_t[df_t['Instance'].str.startswith('planar600')], 'TGraph — planar600 (generalization: larger graph)')
print_summary(df_g[df_g['Instance'].str.startswith('Center')],    'GGraph — Center486 (in-distribution)')
print_summary(df_g[df_g['Instance'].str.startswith('Corners')],   'GGraph — Corners486 (generalization: diff centroid)')
print_summary(df_g[df_g['Instance'].str.startswith('Diagonal')],  'GGraph — Diagonal486 (generalization: diff centroid)')
print_summary(df_all,                                              'ALL INSTANCES')

## 9. Visualization

In [ ]:
# Box plot: Objective per group
groups = [
    ('planar500',    df_t[df_t['Instance'].str.startswith('planar500')]),
    ('planar600',    df_t[df_t['Instance'].str.startswith('planar600')]),
    ('Center486',    df_g[df_g['Instance'].str.startswith('Center')]),
    ('Corners486',   df_g[df_g['Instance'].str.startswith('Corners')]),
    ('Diagonal486',  df_g[df_g['Instance'].str.startswith('Diagonal')]),
]

fig, axes = plt.subplots(1, len(groups), figsize=(16, 5))
for ax, (label, df_sub) in zip(axes, groups):
    if df_sub.empty:
        ax.set_visible(False)
        continue
    bp = ax.boxplot([df_sub['RL_Obj'], df_sub['VNS_Obj']],
                    labels=['BVNS+RL', 'VNS'], patch_artist=True,
                    medianprops=dict(color='red', linewidth=2))
    bp['boxes'][0].set_facecolor('#4a90d9')
    bp['boxes'][1].set_facecolor('#e8a838')
    ax.set_title(label, fontsize=10)
    ax.set_ylabel('Objective (max diameter)')

plt.suptitle('BVNS+RL vs VNS — Objective by instance group', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: RL vs VNS (below y=x → RL better)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, df_sub, title in [
    (axes[0], df_t, 'TGraph (planar500 + planar600)'),
    (axes[1], df_g, 'GGraph (Center + Corners + Diagonal 486)'),
]:
    if df_sub.empty:
        continue
    mn = min(df_sub[['RL_Obj','VNS_Obj']].min())
    mx = max(df_sub[['RL_Obj','VNS_Obj']].max())
    ax.plot([mn, mx], [mn, mx], 'k--', lw=1, label='y = x (equal)')
    ax.scatter(df_sub['VNS_Obj'], df_sub['RL_Obj'],
               c='steelblue', s=55, alpha=0.85, zorder=3)
    ax.fill_between([mn, mx], [mn, mn], [mn, mx],
                    alpha=0.06, color='green', label='RL better')
    ax.fill_between([mn, mx], [mn, mx], [mx, mx],
                    alpha=0.06, color='red', label='VNS better')
    ax.set_xlabel('VNS Objective')
    ax.set_ylabel('BVNS+RL Objective')
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.suptitle('Scatter: BVNS+RL vs VNS Objective', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize solution: TGraph vs GGraph
def plot_solution(graph_path, result_path, ax, title=''):
    G   = nx.read_graphml(graph_path)
    G   = nx.relabel_nodes(G, lambda x: int(x))
    pos = {n: (float(d['x']), float(d['y'])) for n, d in G.nodes(data=True)}
    data = load_json(result_path)
    if data is None:
        ax.set_title(f'{title} (no result yet)')
        return
    districts = {int(k): [int(n) for n in v]
                 for k, v in data['Allocation'].items()}
    palette = cm.get_cmap('tab20')
    nx.draw_networkx_edges(G, pos, ax=ax, width=0.2, alpha=0.3)
    for k, nodes in districts.items():
        nx.draw_networkx_nodes(G, pos, ax=ax, nodelist=nodes,
                               node_color=[palette(k % 20)], node_size=18)
        cx = np.mean([pos[n][0] for n in nodes])
        cy = np.mean([pos[n][1] for n in nodes])
        ax.scatter(cx, cy, color='black', marker='x', s=60, zorder=5)
    ax.set_title(f"{title}\nObj={data['Objective']:.3f}  Inf={data['Infeasibility']:.4f}",
                 fontsize=9)
    ax.axis('off')

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
plot_solution(f'{TGRAPH_INST}/planar500_G0.graphml',
              f'{OUT_PLANAR}/planar500_G0.json',
              axes[0], 'BVNS+RL — planar500_G0')
plot_solution(f'{GGRAPH_INST}/Center486_G0.graphml',
              f'{OUT_GGRAPH}/Center486_G0.json',
              axes[1], 'BVNS+RL — Center486_G0')
plt.tight_layout()
plt.show()